In [1]:
#Modify the path to a directory on your machine
import os
os.environ["CRDS_PATH"] = "/home/hailin/Documents/CRDS"
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"

# Packages that allow us to get information about objects:
import asdf
import copy
import shutil

# Numpy library:
import numpy as np

# For downloading data
import requests

# Astropy tools:
from astropy.io import fits
from astropy.utils.data import download_file
from astropy.visualization import ImageNormalize, ManualInterval, LogStretch

import matplotlib.pyplot as plt
import matplotlib as mpl

# Plotting tools:
from pipeline1_plotting_tools import download_files, plot_jump, plot_jumps, plot_ramp, plot_ramps, show_image, side_by_side

# Use this version for non-interactive plots (easier scrolling of the notebook)
%matplotlib inline

# Use this version (outside of Jupyter Lab) if you want interactive plots
#%matplotlib notebook

# List of possible data quality flags
from jwst.datamodels import dqflags

# The entire calwebb_detector1 pipeline
from jwst.pipeline import calwebb_detector1

# Individual steps that make up calwebb_detector1
from jwst.dq_init import DQInitStep
from jwst.saturation import SaturationStep
from jwst.superbias import SuperBiasStep
from jwst.ipc import IPCStep                                                                                    
from jwst.refpix import RefPixStep                                                                
from jwst.linearity import LinearityStep
from jwst.persistence import PersistenceStep
from jwst.dark_current import DarkCurrentStep
from jwst.jump import JumpStep
from jwst.ramp_fitting import RampFitStep
from jwst import datamodels

import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from pathlib import Path

import jwst
print(jwst.__version__)

1.15.1


In [2]:
data_path = Path('../data/JWST/3000s_exposure')
dark_current = np.zeros((30,3))
i = 0
for item in data_path.iterdir():
    input_file_base = item.name
#    if os.path.exists('./results/blind2/'+ input_file_base +'.txt'):
#        continue
    jump_file = '../data/JWST/3000s_exposure/' + input_file_base + '/' + input_file_base + '_jumpstep.fits'
    jump = datamodels.open(jump_file)
    n_group = jump.data.shape[1]
    photo_start = 0
    photo_end   = n_group - 1
    # bin 的范围
    dn_min = -200
    dn_max = 400
    dn_range = range(dn_min, dn_max)
    total_pix = 2048 * 2048
    print('group number is {}'.format(n_group))

    # flag but no jump from pipeline
    flag_map = (jump.groupdq & ~dqflags.pixel['JUMP_DET'] > 0)
    flag_map_2d = np.sum(flag_map[0, :, :, :], axis=0)
    flag_pix = np.sum(flag_map_2d>0)

    raw_data = jump.data[0]
    pre_mask = jump.data[0,photo_end,:,:] - jump.data[0,photo_start,:,:]

    pvd = np.loadtxt('./results/total/'+ input_file_base + '.txt')

    raw_counts, _ = np.histogram(pre_mask.flatten(), bins=range(dn_min, dn_max+1))

    dark_current[i,0] = np.argmax(raw_counts) + dn_min
    dark_current[i,1] = np.argmax(pvd[:,1]) + dn_min
    peak_position = np.argmax(pvd[:,1])
    dark_current[i,2] = np.sum(pvd[peak_position-50:peak_position+51]) / np.sum(pvd) 


    i += 1
    print(input_file_base)
    print(np.argmax(raw_counts) + dn_min)
    print(np.argmax(pvd[:,1]) + dn_min)


group number is 245
jw01121144001_02102_00001_nrs2
13
5
group number is 245
jw01121156001_02102_00001_nrs2
20
20
group number is 245
jw01121154001_02102_00001_nrs2
16
12
group number is 245
jw01121004001_02102_00001_nrs2
12
8
group number is 245
jw01121116001_02102_00001_nrs2
17
15
group number is 245
jw01121102001_03102_00001_nrs2
20
21
group number is 245
jw01121138001_02102_00001_nrs2
14
10
group number is 245
jw01121128001_02102_00001_nrs2
16
15
group number is 245
jw01121150001_02102_00001_nrs2
12
9
group number is 245
jw01121006001_02102_00001_nrs2
12
7
group number is 245
jw01121110001_02102_00001_nrs2
11
7
group number is 245
jw01121008001_02102_00001_nrs2
10
5
group number is 245
jw01121108001_02102_00001_nrs2
7
3
group number is 245
jw01121142001_02102_00001_nrs2
16
11
group number is 245
jw01121160001_02102_00001_nrs2
14
10
group number is 245
jw01121002001_02102_00001_nrs2
13
8
group number is 245
jw01121106001_02102_00001_nrs2
16
12
group number is 245
jw01121104001_02102_

In [ ]:
np.average(dark_current[:,1])

In [6]:
data_path = Path('../data/JWST/3000s_exposure')
dark_current = np.zeros((30,3))
dn_min = -200
dn_max = 400
dn_range = range(dn_min, dn_max)
i = 0
for item in data_path.iterdir():
    input_file_base = item.name

    pvd = np.loadtxt('./results/total/'+ input_file_base + '.txt')

    dark_current[i,1] = np.argmax(pvd[:,1]) + dn_min
    peak_position = np.argmax(pvd[:,1])
    dark_current[i,2] = np.sum(pvd[peak_position-50:peak_position+51,1]) / np.sum(pvd[:,1]) 


    i += 1
    print(input_file_base)
    print(np.argmax(pvd[:,1]) + dn_min)


jw01121144001_02102_00001_nrs2
5
jw01121156001_02102_00001_nrs2
20
jw01121154001_02102_00001_nrs2
12
jw01121004001_02102_00001_nrs2
8
jw01121116001_02102_00001_nrs2
15
jw01121102001_03102_00001_nrs2
21
jw01121138001_02102_00001_nrs2
10
jw01121128001_02102_00001_nrs2
15
jw01121150001_02102_00001_nrs2
9
jw01121006001_02102_00001_nrs2
7
jw01121110001_02102_00001_nrs2
7
jw01121008001_02102_00001_nrs2
5
jw01121108001_02102_00001_nrs2
3
jw01121142001_02102_00001_nrs2
11
jw01121160001_02102_00001_nrs2
10
jw01121002001_02102_00001_nrs2
8
jw01121106001_02102_00001_nrs2
12
jw01121104001_02102_00001_nrs2
4
jw01121152001_02102_00001_nrs2
27
jw01121136001_02102_00001_nrs2
7
jw01121118001_02102_00001_nrs2
15
jw01121120001_02102_00001_nrs2
3
jw01121134001_02102_00001_nrs2
9
jw01121146001_02102_00001_nrs2
19
jw01121012001_02102_00001_nrs2
0
jw01121114001_02102_00001_nrs2
7
jw01121124001_02102_00001_nrs2
2
jw01121126001_02102_00001_nrs2
10
jw01121130001_02102_00001_nrs2
17
jw01121158001_02102_00001_nrs

In [8]:
np.average(dark_current[:,2])

0.9981319776837504